## Daily Full Funnel Performance Query - All Volume

For all compass and net call steps, gets throughput/breakage/queue rates, and the volume that made it through.

Additionally, have expected conversion and financial plan information. 

Aggregating all data

### Data Aggregation

In [0]:
from typing import Optional, Union, Sequence
from pyspark.sql import DataFrame
from datetime import date, timedelta

def _compute_yday_and_prior4_dates(today: Optional[date] = None):
    if today is None:
        today = date.today()
    yday = today - timedelta(days=1)
    dates = [yday - timedelta(days=7 * i) for i in range(0, 5)]
    return [d.strftime("%Y-%m-%d") for d in dates]


# Columns actually used by compute_kpis()
NEEDED_VCALLS_COLS = [
    "call_date","call_id","company_id","call_direction","spanish_ind",
    "ib_gross_ind","ib_queue_ind","ibs_net_ind","last_ivr_selection_name",
    # used for marketing_bucket + site_serp derivations
    "ivr_split_name","web_session_id",
]

NEEDED_CCF_COLS = [
    "call_id",
    "call_connected_timestamp","compass_start_time","moving_switching_asked_time",
    "movingSwitchingExtracted",
    "return_caller_confirmation_extracted","return_caller_dtmf_boolean","return_caller_confirmation_response",
    "web_address_confirmation_extracted","web_address_confirmation_dtmf_boolean","web_address_confirmation_response",
    "web_zip_confirm_extracted","web_zip_confirmation_dtmf_boolean","web_zip_confirmation_response",
    "web_dob_name_collected_time",
    "zip_code_collected_time","zip_code_confirm_extracted",
    "ss_address_confirmed_time","web_address_confirmed_time","return_caller_confirmed_time",
    "ivrrouting","ercotMatch",
    "name_collected_time","return_caller_path","dob_collected_time",
    "credit_api_start_time","credit_response_status","isCreditHit",
    "first_workflow_leg_id",
    "dead_air_bot","home_business_response",
    "initial_ivr_vf","texas_nontexas_extracted","home_business_extracted",
]

NEEDED_AC_COLS = [
    "call_id",
    "ib_contact_calls","credit_calls_ind","order_count",
]

NEEDED_O_COLS = [
    "call_id",
    "gcv_v2",
]


def get_data(
    marketing_buckets: Optional[Union[str, Sequence[str]]] = None,
    site_serp: Optional[Union[str, Sequence[str]]] = None,
    company_id: Optional[int] = 25,
    call_direction: Optional[str] = "INBOUND",
    restrict_to_yday_and_prior4: bool = True,
    create_temp_view: bool = True,
    print_summary: bool = True,
) -> DataFrame:
    """
    Faster get_data(): only selects columns required by compute_kpis()
    (plus columns needed to derive marketing_bucket/site_serp).

    Returns a wide calls_full DataFrame and optionally creates TEMP VIEW calls_full.
    """

    # ---------------------------
    # Normalize & validate filters (same behavior as before)
    # ---------------------------
    allowed_buckets = {
        "Natural","Brand-Partner","Generic","Aggregator","Competitor","Utility","PMax","NRG","Other Bucket"
    }

    def _as_list(x):
        if x is None:
            return None
        if isinstance(x, str):
            return [x]
        return list(x)

    mb_list = _as_list(marketing_buckets)
    if mb_list is not None:
        norm_map = {
            "brandpartner": "Brand-Partner",
            "brand-partner": "Brand-Partner",
            "brand_partner": "Brand-Partner",
            "pmax": "PMax",
            "nrg": "NRG",
            "other": "Other Bucket",
            "other bucket": "Other Bucket",
            "utility": "Utility",
            "natural": "Natural",
            "generic": "Generic",
            "aggregator": "Aggregator",
            "competitor": "Competitor",
        }
        mb_list = [norm_map.get(b.strip().lower(), b) for b in mb_list]
        unknown = [b for b in mb_list if b not in allowed_buckets]
        if unknown:
            raise ValueError(f"Unknown marketing_buckets: {unknown}. Allowed: {sorted(allowed_buckets)}")

    ss_list = _as_list(site_serp)
    if ss_list is not None:
        norm_ss = {"site": "Site", "serp": "SERP"}
        ss_list = [norm_ss.get(s.strip().lower(), s) for s in ss_list]
        unknown = [s for s in ss_list if s not in {"Site", "SERP"}]
        if unknown:
            raise ValueError("site_serp must be 'Site', 'SERP', or a list of those values.")

    # ---------------------------
    # Source tables
    # ---------------------------
    base_table = "energy_prod.energy.v_calls"
    ccf_table  = "ai_products_prod.energy.compass_call_fct"
    ac_table   = "energy_prod.energy.v_agent_calls"
    o_table    = "energy_prod.energy.v_orders"

    # ---------------------------
    # CASE expressions (unchanged)
    # ---------------------------
    marketing_bucket_case = """
      CASE
        WHEN c.ivr_split_name IN ('natural_marketingbucket','natural_marketingbucket_serp') THEN 'Natural'
        WHEN c.ivr_split_name IN ('brandpartner_marketingbucket_serp','brandpartner_marketingbucket') THEN 'Brand-Partner'
        WHEN c.ivr_split_name IN ('generic_marketingbucket_serp','generic_marketingbucket') THEN 'Generic'
        WHEN c.ivr_split_name IN ('aggregator_marketingbucket_serp','aggregator_marketingbucket') THEN 'Aggregator'
        WHEN c.ivr_split_name IN ('competitor_marketingbucket_serp','competitor_marketingbucket') THEN 'Competitor'
        WHEN c.ivr_split_name IN ('dereg_utility_check_serp','dereg_utility_check') THEN 'Utility'
        WHEN c.ivr_split_name IN ('pmax_marketingbucket_serp','pmax_marketingbucket') THEN 'PMax'
        WHEN c.ivr_split_name IN ('nrg_bucket_serp','nrg_bucket') THEN 'NRG'
        ELSE 'Other Bucket'
      END
    """

    site_serp_case = """
      CASE
        WHEN c.web_session_id IS NOT NULL THEN 'Site'
        ELSE 'SERP'
      END
    """

    # ---------------------------
    # WHERE (base)
    # ---------------------------
    base_where_clauses = []
    if company_id is not None:
        base_where_clauses.append(f"c.company_id = {int(company_id)}")
    if call_direction is not None:
        safe_call_dir = str(call_direction).replace("'", "''")
        base_where_clauses.append(f"UPPER(c.call_direction) = UPPER('{safe_call_dir}')")

    pulled_dates = None
    if restrict_to_yday_and_prior4:
        pulled_dates = _compute_yday_and_prior4_dates()
        dates_sql = ", ".join([f"'{d}'" for d in pulled_dates])
        base_where_clauses.append(f"c.call_date IN ({dates_sql})")

    base_where_sql = "\n        AND ".join(base_where_clauses) if base_where_clauses else "1=1"

    # post filters (marketing bucket / site_serp)
    post_filters = []
    if mb_list:
        mb_sql = ", ".join([f"'{b}'" for b in mb_list])
        post_filters.append(f"marketing_bucket IN ({mb_sql})")
    if ss_list:
        ss_sql = ", ".join([f"'{s}'" for s in ss_list])
        post_filters.append(f"site_serp IN ({ss_sql})")

    post_where_sql = ""
    if post_filters:
        post_where_sql = "WHERE " + " AND ".join(post_filters)

    # ---------------------------
    # SELECT lists (only required cols)
    # ---------------------------
    # v_calls needed + derived fields
    base_select_cols = ", ".join([f"c.`{col}`" for col in NEEDED_VCALLS_COLS])

    # ccf/ac/o needed, with prefixes to match compute_kpis expectations
    ccf_select_cols = ",\n      ".join(
        [f"ccf.`{col}` AS `ccf_{col}`" for col in NEEDED_CCF_COLS if col != "call_id"]
    )
    ac_select_cols = ",\n      ".join(
        [f"ac.`{col}` AS `ac_{col}`" for col in NEEDED_AC_COLS if col != "call_id"]
    )
    o_select_cols = ",\n      ".join(
        [f"o.`{col}` AS `o_{col}`" for col in NEEDED_O_COLS if col != "call_id"]
    )

    # final projection order
    final_select_parts = [
        # base calls (including ivr_split_name/web_session_id needed for derivations)
        ", ".join([f"c.`{col}`" for col in NEEDED_VCALLS_COLS]),
        "c.marketing_bucket",
        "c.site_serp",
        ccf_select_cols,
        ac_select_cols,
        o_select_cols,
    ]
    final_select_sql = ",\n      ".join([p for p in final_select_parts if p.strip()])

    calls_full_sql = f"""
    WITH base_calls AS (
      SELECT
        {base_select_cols},
        {marketing_bucket_case} AS marketing_bucket,
        {site_serp_case} AS site_serp
      FROM {base_table} c
      WHERE {base_where_sql}
    ),
    filtered_calls AS (
      SELECT *
      FROM base_calls
      {post_where_sql}
    )
    SELECT
      {final_select_sql}
    FROM filtered_calls c
    LEFT JOIN {ccf_table} ccf
      ON c.call_id = ccf.call_id
    LEFT JOIN {ac_table} ac
      ON c.call_id = ac.call_id
    LEFT JOIN {o_table} o
      ON c.call_id = o.call_id
    """

    df = spark.sql(calls_full_sql)

    if create_temp_view:
        df.createOrReplaceTempView("calls_full")

    if print_summary:
        if pulled_dates is not None:
            print(f"get_data(): pulled call_date IN {pulled_dates}")
        else:
            print("get_data(): no call_date restriction applied")
        print(f"get_data(): rows = {df.count()}")
        if create_temp_view:
            print("get_data(): created TEMP VIEW calls_full")

    return df


### Full Funnel Query

In [0]:
from typing import Optional
from pyspark.sql import DataFrame

def compute_kpis(
    calls_df: Optional[DataFrame] = None,
    source_view: str = "calls_full",
    create_temp_view: bool = True,
    debug_print_sql: bool = True,
) -> DataFrame:
    """
    Serverless-friendly compute_kpis where the full KPI SQL is embedded in the function.

    - If calls_df is provided and create_temp_view=True, it will createOrReplaceTempView(source_view).
    - The internal SQL references the given source_view and creates a temp view named out_view.
    - Returns a DataFrame from SELECT * FROM out_view.
    """

    # If caller provided a DataFrame, register it as the source view
    if calls_df is not None and create_temp_view:
        calls_df.createOrReplaceTempView(source_view)

    # Build a safe output view name (unique-ish)
    safe_view = (
        source_view.replace("`", "")
        .replace(".", "_")
        .replace("-", "_")
        .replace(" ", "_")
        .replace("|", "_")
        .replace("/", "_")
    )
    out_view = f"full_funnel_kpis__{safe_view}"

    # Embedded SQL (no date-filtering here; computation expects the source_view to already be limited to relevant dates)
    full_funnel_kpis_sql = f"""
CREATE OR REPLACE TEMP VIEW {out_view} AS
WITH params AS (
  SELECT
    date_sub(current_date(), 1)            AS yday,
    dayofweek(date_sub(current_date(), 1)) AS yday_dow,
    date_format(date_sub(current_date(), 1), 'yyyy-MM-dd') AS yday_label
),

base AS (
  SELECT
    cf.call_date,
    cf.call_id,

    -- v_calls (no prefix in calls_full)
    cf.company_id,
    cf.call_direction,
    cf.spanish_ind,
    cf.ib_gross_ind,
    cf.ib_queue_ind,
    cf.ibs_net_ind,

    -- v_agent_calls (ac_)
    cf.ac_ib_contact_calls,
    cf.ac_credit_calls_ind,
    cf.ac_order_count,

    -- v_orders (o_)
    cf.o_gcv_v2,

    -- Entered Compass + key step flags
    CASE
      WHEN try_cast(cf.ccf_call_connected_timestamp AS timestamp) IS NOT NULL THEN 1
      WHEN try_cast(cf.ccf_compass_start_time AS timestamp) IS NOT NULL THEN 1
      ELSE 0
    END AS entered_compass,

    CASE
      WHEN try_cast(cf.ccf_moving_switching_asked_time AS timestamp) IS NOT NULL THEN 1
      ELSE 0
    END AS moving_switching_asked,

    CASE
      WHEN cf.ccf_movingSwitchingExtracted IS NOT NULL
           AND cf.ccf_movingSwitchingExtracted NOT IN ('0','null') THEN 1
      WHEN cf.ccf_return_caller_confirmation_extracted IN ('Yes','yes')
           OR (cf.ccf_return_caller_dtmf_boolean = 'dtmf'
               AND cf.ccf_return_caller_confirmation_response IN (1,11)) THEN 1
      WHEN cf.ccf_web_address_confirmation_extracted IN ('Yes','yes')
           OR (cf.ccf_web_address_confirmation_dtmf_boolean = 'dtmf'
               AND cf.ccf_web_address_confirmation_response IN (1,11)) THEN 1
      WHEN cf.ccf_web_zip_confirm_extracted IN ('Yes','yes')
           OR (cf.ccf_web_zip_confirmation_dtmf_boolean = 'dtmf'
               AND cf.ccf_web_zip_confirmation_response IN (1,11)) THEN 1
      WHEN try_cast(cf.ccf_web_dob_name_collected_time AS timestamp) IS NOT NULL THEN 1
      ELSE 0
    END AS moving_switching_extracted,

    CASE
      WHEN try_cast(cf.ccf_zip_code_collected_time AS timestamp) IS NOT NULL THEN 1
      WHEN cf.ccf_zip_code_confirm_extracted IN ('Yes','yes') THEN 1
      WHEN cf.ccf_web_address_confirmation_extracted IN ('Yes','yes')
           OR (cf.ccf_web_address_confirmation_dtmf_boolean = 'dtmf'
               AND cf.ccf_web_address_confirmation_response IN (1,11)) THEN 1
      WHEN cf.ccf_web_zip_confirm_extracted IN ('Yes','yes')
           OR (cf.ccf_web_zip_confirmation_dtmf_boolean = 'dtmf'
               AND cf.ccf_web_zip_confirmation_response IN (1,11)) THEN 1
      WHEN cf.ccf_return_caller_confirmation_extracted IN ('Yes','yes')
           OR (cf.ccf_return_caller_dtmf_boolean = 'dtmf'
               AND cf.ccf_return_caller_confirmation_response IN (1,11)) THEN 1
      WHEN try_cast(cf.ccf_web_dob_name_collected_time AS timestamp) IS NOT NULL THEN 1
      ELSE 0
    END AS any_zip_collected,

    CASE
      WHEN try_cast(cf.ccf_ss_address_confirmed_time AS timestamp) IS NOT NULL THEN 1
      WHEN try_cast(cf.ccf_web_address_confirmed_time AS timestamp) IS NOT NULL THEN 1
      WHEN cf.ccf_return_caller_confirmed_time IN ('Yes','yes') THEN 1
      ELSE 0
    END AS any_address_collected,

    CASE
      WHEN cf.ccf_ivrrouting = 'default'
       AND cf.ccf_ercotMatch = 'true'
       AND (
            try_cast(cf.ccf_ss_address_confirmed_time AS timestamp) IS NOT NULL
         OR try_cast(cf.ccf_web_address_confirmed_time AS timestamp) IS NOT NULL
         OR cf.ccf_return_caller_confirmed_time IN ('Yes','yes')
       )
      THEN 1 ELSE 0
    END AS address_collected_matched,

    CASE
      WHEN try_cast(cf.ccf_name_collected_time AS timestamp) IS NOT NULL THEN 1
      WHEN try_cast(cf.ccf_web_dob_name_collected_time AS timestamp) IS NOT NULL THEN 1
      WHEN cf.ccf_return_caller_path = 'credit' THEN 1
      ELSE 0
    END AS name_confirmed,

    CASE
      WHEN try_cast(cf.ccf_dob_collected_time AS timestamp) IS NOT NULL THEN 1
      WHEN try_cast(cf.ccf_web_dob_name_collected_time AS timestamp) IS NOT NULL THEN 1
      WHEN cf.ccf_return_caller_path = 'credit' THEN 1
      ELSE 0
    END AS dob_collected,

    CASE
      WHEN try_cast(cf.ccf_credit_api_start_time AS timestamp) IS NOT NULL THEN 1
      ELSE 0
    END AS ucc_api_call,

    CASE
      WHEN cf.ccf_credit_response_status = 'Successful' THEN 1
      ELSE 0
    END AS ucc_api_call_successful,

    CASE
      WHEN cf.ccf_isCreditHit = 'true' THEN 1
      WHEN cf.ccf_return_caller_confirmed_time IN ('Yes','yes')
           AND cf.ccf_return_caller_path = 'credit' THEN 1
      ELSE 0
    END AS ucc_credit_hit,

    CASE
      WHEN cf.ccf_first_workflow_leg_id IS NOT NULL THEN 1
      ELSE 0
    END AS compass_completion,

    -- Bot flags
    CASE WHEN cf.ccf_dead_air_bot = '1' THEN 1 ELSE 0 END AS deadair_bot_traffic,

    CASE
      WHEN cf.ccf_home_business_response RLIKE
        '(^|[^0-9])' ||
        '([+]?[[:space:]().-]*1[[:space:]().-]*)?' ||
        '(' ||
          '8[0-9]{{2}}[[:space:]().-]*[0-9]{{3}}[[:space:]().-]*[0-9]{{4}}' ||
          '|' ||
          '8[0-9]{{2}}-[0-9]{{3}}' ||
          '|' ||
          '8[0-9]{{2}}-[0-9]{{4}}' ||
        ')' ||
        '([^0-9]|$)'
      THEN 1 ELSE 0
    END AS phone_number_bot_traffic,

    CASE
      WHEN cf.ccf_home_business_response RLIKE '([Pp])ress[[:space:]]+[0-9]'
      THEN 1 ELSE 0
    END AS press_number_bot_traffic,

    CASE
      WHEN (cf.ccf_dead_air_bot = '1')
        OR (cf.ccf_home_business_response RLIKE
          '(^|[^0-9])' ||
          '([+]?[[:space:]().-]*1[[:space:]().-]*)?' ||
          '(' ||
            '8[0-9]{{2}}[[:space:]().-]*[0-9]{{3}}[[:space:]().-]*[0-9]{{4}}' ||
            '|' ||
            '8[0-9]{{2}}-[0-9]{{3}}' ||
            '|' ||
            '8[0-9]{{2}}-[0-9]{{4}}' ||
          ')' ||
          '([^0-9]|$)')
        OR (cf.ccf_home_business_response RLIKE '([Pp])ress[[:space:]]+[0-9]')
      THEN 1 ELSE 0
    END AS any_bot_call,

    -- IVR completion proxy
    CASE
      WHEN cf.last_ivr_selection_name IN ('TX','TX-1') THEN 1
      WHEN cf.ccf_initial_ivr_vf = 1 AND cf.ccf_texas_nontexas_extracted = 'Texas' THEN 1
      WHEN cf.ccf_initial_ivr_vf = 1
           AND cf.ccf_home_business_extracted = 'residential'
           AND cf.call_date < DATE '2025-09-24' THEN 1
      WHEN try_cast(cf.ccf_moving_switching_asked_time AS timestamp) IS NOT NULL THEN 1
      WHEN cf.ccf_return_caller_confirmation_extracted IN ('Yes','yes')
           OR (cf.ccf_return_caller_dtmf_boolean = 'dtmf'
               AND cf.ccf_return_caller_confirmation_response IN (1,11)) THEN 1
      WHEN cf.ccf_web_address_confirmation_extracted IN ('Yes','yes')
           OR (cf.ccf_web_address_confirmation_dtmf_boolean = 'dtmf'
               AND cf.ccf_web_address_confirmation_response IN (1,11)) THEN 1
      WHEN cf.ccf_web_zip_confirm_extracted IN ('Yes','yes')
           OR (cf.ccf_web_zip_confirmation_dtmf_boolean = 'dtmf'
               AND cf.ccf_web_zip_confirmation_response IN (1,11)) THEN 1
      WHEN try_cast(cf.ccf_web_dob_name_collected_time AS timestamp) IS NOT NULL THEN 1
      ELSE 0
    END AS ivr_completed_call,

    -- STEP REACH denominators
    CASE WHEN entered_compass = 1 THEN 1 ELSE 0 END AS reached_entered_compass,
    CASE WHEN entered_compass = 1 AND ivr_completed_call = 1 THEN 1 ELSE 0 END AS reached_moving_switching,
    CASE WHEN entered_compass = 1 AND moving_switching_extracted = 1 THEN 1 ELSE 0 END AS reached_zip_collection,
    CASE WHEN entered_compass = 1 AND any_zip_collected = 1 THEN 1 ELSE 0 END AS reached_address_collection,
    CASE WHEN entered_compass = 1 AND any_address_collected = 1 THEN 1 ELSE 0 END AS reached_ercot_check,
    CASE WHEN entered_compass = 1 AND address_collected_matched = 1 THEN 1 ELSE 0 END AS reached_name_collection,
    CASE WHEN entered_compass = 1 AND name_confirmed = 1 THEN 1 ELSE 0 END AS reached_dob_collection,
    CASE WHEN entered_compass = 1 AND dob_collected = 1 THEN 1 ELSE 0 END AS reached_credit_check,

    -- QUEUE @ STEP
    CASE WHEN entered_compass = 1 AND cf.ib_queue_ind = 1 AND ivr_completed_call = 0 THEN 1 ELSE 0 END AS q_at_initial_question,
    CASE WHEN cf.ib_queue_ind = 1 AND cf.ccf_ivrrouting = 'default' AND moving_switching_extracted = 0 THEN 1 ELSE 0 END AS q_at_moving_switching,
    CASE WHEN cf.ib_queue_ind = 1 AND cf.ccf_ivrrouting = 'default' AND moving_switching_extracted = 1 AND any_zip_collected = 0 THEN 1 ELSE 0 END AS q_at_zip_collection,
    CASE WHEN cf.ib_queue_ind = 1 AND cf.ccf_ivrrouting = 'default' AND any_zip_collected = 1 AND any_address_collected = 0 THEN 1 ELSE 0 END AS q_at_address_collection,
    CASE WHEN cf.ib_queue_ind = 1 AND cf.ccf_ivrrouting = 'default' AND any_address_collected = 1 AND address_collected_matched = 0 THEN 1 ELSE 0 END AS q_at_ercot_check,
    CASE WHEN cf.ib_queue_ind = 1 AND cf.ccf_ivrrouting = 'default' AND address_collected_matched = 1 AND name_confirmed = 0 THEN 1 ELSE 0 END AS q_at_name_collection,
    CASE WHEN cf.ib_queue_ind = 1 AND cf.ccf_ivrrouting = 'default' AND name_confirmed = 1 AND dob_collected = 0 THEN 1 ELSE 0 END AS q_at_dob_collection,
    CASE WHEN cf.ib_queue_ind = 1 AND cf.ccf_ivrrouting = 'default' AND dob_collected = 1 AND ucc_credit_hit = 0 THEN 1 ELSE 0 END AS q_at_credit_check_no_hit,
    CASE WHEN cf.ib_queue_ind = 1 AND cf.ccf_ivrrouting = 'default' AND ucc_credit_hit = 1 THEN 1 ELSE 0 END AS q_with_credit_hit,

    -- BREAKAGE @ STEP
    CASE WHEN entered_compass = 1 AND cf.ib_queue_ind = 0 AND ivr_completed_call = 0 THEN 1 ELSE 0 END AS b_at_initial_question,
    CASE WHEN cf.ib_queue_ind = 0 AND moving_switching_extracted = 0 AND ivr_completed_call = 1 THEN 1 ELSE 0 END AS b_at_moving_switching,
    CASE WHEN cf.ib_queue_ind = 0 AND moving_switching_extracted = 1
              AND try_cast(cf.ccf_moving_switching_asked_time AS timestamp) IS NOT NULL
              AND any_zip_collected = 0
         THEN 1 ELSE 0 END AS b_at_zip_collection,
    CASE WHEN cf.ib_queue_ind = 0 AND any_zip_collected = 1 AND any_address_collected = 0 THEN 1 ELSE 0 END AS b_at_address_collection,
    CASE WHEN cf.ib_queue_ind = 0 AND any_address_collected = 1 AND address_collected_matched = 0 THEN 1 ELSE 0 END AS b_at_ercot_check,
    CASE WHEN cf.ib_queue_ind = 0 AND address_collected_matched = 1 AND name_confirmed = 0 THEN 1 ELSE 0 END AS b_at_name_collection,
    CASE WHEN cf.ib_queue_ind = 0 AND name_confirmed = 1 AND dob_collected = 0 THEN 1 ELSE 0 END AS b_at_dob_collection,
    CASE WHEN cf.ib_queue_ind = 0 AND dob_collected = 1 AND ucc_credit_hit = 0 THEN 1 ELSE 0 END AS b_at_credit_check_no_hit,
    CASE WHEN cf.ib_queue_ind = 0 AND ucc_credit_hit = 1 THEN 1 ELSE 0 END AS b_with_credit_hit

  FROM {source_view} cf
  CROSS JOIN params p
  WHERE cf.call_date >= date_sub(p.yday, 60)
    AND cf.call_date <= p.yday
    AND dayofweek(cf.call_date) = p.yday_dow
    AND cf.spanish_ind = 0
    AND cf.company_id = 25
    AND cf.call_direction IN ('Inbound','INBOUND')
),

prior4_dates AS (
  SELECT call_date
  FROM (
    SELECT
      call_date,
      dense_rank() OVER (ORDER BY call_date DESC) AS dr
    FROM (SELECT DISTINCT call_date FROM base)
    CROSS JOIN params p
    WHERE call_date < p.yday
  ) t
  WHERE dr <= 4
),

scoped AS (
  SELECT
    b.*,
    CASE
      WHEN b.call_date = p.yday THEN 'yesterday'
      WHEN b.call_date IN (SELECT call_date FROM prior4_dates) THEN 'prior4'
      ELSE 'ignore'
    END AS period
  FROM base b
  CROSS JOIN params p
),

daily AS (
  SELECT
    call_date,
    period,

    -- Core volumes
    SUM(CASE WHEN ib_gross_ind = 1 THEN 1 ELSE 0 END) AS gross_calls,

    SUM(CASE WHEN ib_gross_ind = 1 THEN entered_compass ELSE 0 END) AS entered_compass_calls,
    try_divide(
      SUM(CASE WHEN ib_gross_ind = 1 THEN entered_compass ELSE 0 END),
      NULLIF(SUM(CASE WHEN ib_gross_ind = 1 THEN 1 ELSE 0 END), 0)
    ) AS entered_compass_rate,

    try_divide(
      SUM(compass_completion),
      NULLIF(SUM(entered_compass), 0)
    ) AS compass_completion_rate,

    -- Bot KPIs (Compass-scoped)
    try_divide(
      SUM(CASE WHEN entered_compass = 1 AND any_bot_call = 1 THEN 1 ELSE 0 END),
      NULLIF(SUM(CASE WHEN entered_compass = 1 THEN 1 ELSE 0 END), 0)
    ) AS any_bot_call_rate_compass,

    ( SUM(CASE WHEN entered_compass = 1 THEN 1 ELSE 0 END)
      - SUM(CASE WHEN entered_compass = 1 AND any_bot_call = 1 THEN 1 ELSE 0 END)
    ) AS non_bot_compass_calls,

    -- Queue metrics
    SUM(CASE WHEN ib_queue_ind = 1 THEN 1 ELSE 0 END) AS queue_calls,
    try_divide(
      SUM(CASE WHEN ib_queue_ind = 1 THEN 1 ELSE 0 END),
      NULLIF(SUM(CASE WHEN ib_gross_ind = 1 THEN 1 ELSE 0 END), 0)
    ) AS queue_to_gross_rate,

    try_divide(
      SUM(CASE WHEN entered_compass = 1 THEN ivr_completed_call ELSE 0 END),
      NULLIF(SUM(CASE WHEN ib_queue_ind = 1 THEN 1 ELSE 0 END), 0)
    ) AS queue_to_ivr_rate,

    try_divide(
      SUM(CASE WHEN ib_queue_ind = 1 AND ibs_net_ind = 0 THEN 1 ELSE 0 END),
      NULLIF(SUM(CASE WHEN ib_gross_ind = 1 THEN 1 ELSE 0 END), 0)
    ) AS abandonment_rate,

    -- Net + downstream rates
    SUM(CASE WHEN ibs_net_ind = 1 THEN 1 ELSE 0 END) AS net_calls,

    try_divide(
      SUM(CASE WHEN ibs_net_ind = 1 AND ac_ib_contact_calls = 1 THEN 1 ELSE 0 END),
      NULLIF(SUM(CASE WHEN ibs_net_ind = 1 THEN 1 ELSE 0 END), 0)
    ) AS contact_rate,

    try_divide(
      SUM(CASE WHEN ibs_net_ind = 1 AND ac_credit_calls_ind = 1 THEN 1 ELSE 0 END),
      NULLIF(SUM(CASE WHEN ibs_net_ind = 1 THEN 1 ELSE 0 END), 0)
    ) AS credit_rate,

    try_divide(
      SUM(CASE WHEN ibs_net_ind = 1 AND ac_order_count = 1 THEN 1 ELSE 0 END),
      NULLIF(SUM(CASE WHEN ibs_net_ind = 1 THEN 1 ELSE 0 END), 0)
    ) AS net_conversion_rate,

    SUM(CASE WHEN ac_order_count = 1 THEN 1 ELSE 0 END) AS orders,

    try_divide(
      SUM(CASE WHEN ac_order_count = 1 THEN 1 ELSE 0 END),
      NULLIF(SUM(CASE WHEN ib_gross_ind = 1 THEN 1 ELSE 0 END), 0)
    ) AS gross_conversion_rate,

    try_divide(
      SUM(CASE WHEN ac_order_count = 1 THEN 1 ELSE 0 END),
      NULLIF(
        ( SUM(CASE WHEN entered_compass = 1 THEN 1 ELSE 0 END)
          - SUM(CASE WHEN entered_compass = 1 AND any_bot_call = 1 THEN 1 ELSE 0 END)
        ),
        0
      )
    ) AS transactional_conversion_rate,

    -- UCC operational rates
    try_divide(SUM(ucc_api_call),            NULLIF(SUM(entered_compass), 0)) AS ucc_api_call_rate,
    try_divide(SUM(ucc_api_call_successful), NULLIF(SUM(ucc_api_call), 0))    AS ucc_api_success_given_call_rate,
    try_divide(SUM(ucc_credit_hit),          NULLIF(SUM(ucc_api_call_successful), 0)) AS ucc_hit_given_success_rate,

    -- Step loss rates
    try_divide(SUM(q_at_initial_question), NULLIF(SUM(reached_entered_compass), 0)) AS q_rate_initial_question,
    try_divide(SUM(b_at_initial_question), NULLIF(SUM(reached_entered_compass), 0)) AS b_rate_initial_question,

    try_divide(SUM(q_at_moving_switching), NULLIF(SUM(reached_moving_switching), 0)) AS q_rate_moving_switching,
    try_divide(SUM(b_at_moving_switching), NULLIF(SUM(reached_moving_switching), 0)) AS b_rate_moving_switching,

    try_divide(SUM(q_at_zip_collection), NULLIF(SUM(reached_zip_collection), 0)) AS q_rate_zip_collection,
    try_divide(SUM(b_at_zip_collection), NULLIF(SUM(reached_zip_collection), 0)) AS b_rate_zip_collection,

    try_divide(SUM(q_at_address_collection), NULLIF(SUM(reached_address_collection), 0)) AS q_rate_address_collection,
    try_divide(SUM(b_at_address_collection), NULLIF(SUM(reached_address_collection), 0)) AS b_rate_address_collection,

    try_divide(SUM(q_at_ercot_check), NULLIF(SUM(reached_ercot_check), 0)) AS q_rate_ercot_check,
    try_divide(SUM(b_at_ercot_check), NULLIF(SUM(reached_ercot_check), 0)) AS b_rate_ercot_check,

    try_divide(SUM(q_at_name_collection), NULLIF(SUM(reached_name_collection), 0)) AS q_rate_name_collection,
    try_divide(SUM(b_at_name_collection), NULLIF(SUM(reached_name_collection), 0)) AS b_rate_name_collection,

    try_divide(SUM(q_at_dob_collection), NULLIF(SUM(reached_dob_collection), 0)) AS q_rate_dob_collection,
    try_divide(SUM(b_at_dob_collection), NULLIF(SUM(reached_dob_collection), 0)) AS b_rate_dob_collection,

    try_divide(SUM(q_at_credit_check_no_hit), NULLIF(SUM(reached_credit_check), 0)) AS q_rate_credit_check_no_hit,
    try_divide(SUM(b_at_credit_check_no_hit), NULLIF(SUM(reached_credit_check), 0)) AS b_rate_credit_check_no_hit,

    try_divide(SUM(q_with_credit_hit), NULLIF(SUM(reached_credit_check), 0)) AS q_rate_with_credit_hit,
    try_divide(SUM(b_with_credit_hit), NULLIF(SUM(reached_credit_check), 0)) AS b_rate_with_credit_hit,

    -- Step counts (unchanged)
    SUM(reached_entered_compass) AS reached_entered_compass_calls,
    SUM(reached_moving_switching) AS reached_moving_switching_calls,
    SUM(reached_zip_collection) AS reached_zip_collection_calls,
    SUM(reached_address_collection) AS reached_address_collection_calls,
    SUM(reached_ercot_check) AS reached_ercot_check_calls,
    SUM(reached_name_collection) AS reached_name_collection_calls,
    SUM(reached_dob_collection) AS reached_dob_collection_calls,
    SUM(reached_credit_check) AS reached_credit_check_calls,

    SUM(q_at_initial_question) AS q_at_initial_question_calls,
    SUM(b_at_initial_question) AS b_at_initial_question_calls,
    SUM(q_at_moving_switching) AS q_at_moving_switching_calls,
    SUM(b_at_moving_switching) AS b_at_moving_switching_calls,
    SUM(q_at_zip_collection) AS q_at_zip_collection_calls,
    SUM(b_at_zip_collection) AS b_at_zip_collection_calls,
    SUM(q_at_address_collection) AS q_at_address_collection_calls,
    SUM(b_at_address_collection) AS b_at_address_collection_calls,
    SUM(q_at_ercot_check) AS q_at_ercot_check_calls,
    SUM(b_at_ercot_check) AS b_at_ercot_check_calls,
    SUM(q_at_name_collection) AS q_at_name_collection_calls,
    SUM(b_at_name_collection) AS b_at_name_collection_calls,
    SUM(q_at_dob_collection) AS q_at_dob_collection_calls,
    SUM(b_at_dob_collection) AS b_at_dob_collection_calls,
    SUM(q_at_credit_check_no_hit) AS q_at_credit_check_no_hit_calls,
    SUM(b_at_credit_check_no_hit) AS b_at_credit_check_no_hit_calls,
    SUM(q_with_credit_hit) AS q_with_credit_hit_calls,
    SUM(b_with_credit_hit) AS b_with_credit_hit_calls,

    SUM(CASE WHEN entered_compass = 1 THEN ivr_completed_call ELSE 0 END) AS ivr_completed_calls,

    SUM(CASE WHEN ac_order_count = 1 THEN CAST(o_gcv_v2 AS DOUBLE) ELSE 0.0 END) AS total_revenue,

    try_divide(
      SUM(CASE WHEN ac_order_count = 1 THEN CAST(o_gcv_v2 AS DOUBLE) ELSE 0.0 END),
      NULLIF(SUM(CASE WHEN ac_order_count = 1 THEN 1 ELSE 0 END), 0)
    ) AS revenue_per_order,

    try_divide(
      SUM(CASE WHEN ac_order_count = 1 THEN CAST(o_gcv_v2 AS DOUBLE) ELSE 0.0 END),
      NULLIF(SUM(CASE WHEN ibs_net_ind = 1 THEN 1 ELSE 0 END), 0)
    ) AS revenue_per_net_call,

    try_divide(
      SUM(CASE WHEN ac_order_count = 1 THEN CAST(o_gcv_v2 AS DOUBLE) ELSE 0.0 END),
      NULLIF(SUM(CASE WHEN ib_gross_ind = 1 THEN 1 ELSE 0 END), 0)
    ) AS revenue_per_gross_call

  FROM scoped
  WHERE period IN ('yesterday','prior4')
  GROUP BY call_date, period
),

period_agg AS (
  SELECT
    MAX(p.yday_label) AS yday_label,

    AVG(CASE WHEN period='prior4' THEN CAST(gross_calls AS DOUBLE) END) AS p4_gross_calls,
    AVG(CASE WHEN period='prior4' THEN entered_compass_rate END) AS p4_entered_compass_rate,
    AVG(CASE WHEN period='prior4' THEN CAST(entered_compass_calls AS DOUBLE) END) AS p4_entered_compass_calls,
    AVG(CASE WHEN period='prior4' THEN compass_completion_rate END) AS p4_compass_completion_rate,

    AVG(CASE WHEN period='prior4' THEN any_bot_call_rate_compass END) AS p4_any_bot_call_rate_compass,
    AVG(CASE WHEN period='prior4' THEN CAST(non_bot_compass_calls AS DOUBLE) END) AS p4_non_bot_compass_calls,

    AVG(CASE WHEN period='prior4' THEN queue_to_gross_rate END) AS p4_queue_to_gross_rate,
    AVG(CASE WHEN period='prior4' THEN CAST(queue_calls AS DOUBLE) END) AS p4_queue_calls,
    AVG(CASE WHEN period='prior4' THEN abandonment_rate END) AS p4_abandonment_rate,
    AVG(CASE WHEN period='prior4' THEN CAST(net_calls AS DOUBLE) END) AS p4_net_calls,

    AVG(CASE WHEN period='prior4' THEN contact_rate END) AS p4_contact_rate,
    AVG(CASE WHEN period='prior4' THEN credit_rate END) AS p4_credit_rate,
    AVG(CASE WHEN period='prior4' THEN net_conversion_rate END) AS p4_net_conversion_rate,
    AVG(CASE WHEN period='prior4' THEN CAST(orders AS DOUBLE) END) AS p4_orders,
    AVG(CASE WHEN period='prior4' THEN gross_conversion_rate END) AS p4_gross_conversion_rate,

    AVG(CASE WHEN period='prior4' THEN ucc_api_call_rate END) AS p4_ucc_api_call_rate,
    AVG(CASE WHEN period='prior4' THEN ucc_api_success_given_call_rate END) AS p4_ucc_api_success_given_call_rate,
    AVG(CASE WHEN period='prior4' THEN ucc_hit_given_success_rate END) AS p4_ucc_hit_given_success_rate,

    AVG(CASE WHEN period='prior4' THEN q_rate_initial_question END) AS p4_q_rate_initial_question,
    AVG(CASE WHEN period='prior4' THEN b_rate_initial_question END) AS p4_b_rate_initial_question,

    AVG(CASE WHEN period='prior4' THEN q_rate_moving_switching END) AS p4_q_rate_moving_switching,
    AVG(CASE WHEN period='prior4' THEN b_rate_moving_switching END) AS p4_b_rate_moving_switching,

    AVG(CASE WHEN period='prior4' THEN q_rate_zip_collection END) AS p4_q_rate_zip_collection,
    AVG(CASE WHEN period='prior4' THEN b_rate_zip_collection END) AS p4_b_rate_zip_collection,

    AVG(CASE WHEN period='prior4' THEN q_rate_address_collection END) AS p4_q_rate_address_collection,
    AVG(CASE WHEN period='prior4' THEN b_rate_address_collection END) AS p4_b_rate_address_collection,

    AVG(CASE WHEN period='prior4' THEN q_rate_ercot_check END) AS p4_q_rate_ercot_check,
    AVG(CASE WHEN period='prior4' THEN b_rate_ercot_check END) AS p4_b_rate_ercot_check,

    AVG(CASE WHEN period='prior4' THEN q_rate_name_collection END) AS p4_q_rate_name_collection,
    AVG(CASE WHEN period='prior4' THEN b_rate_name_collection END) AS p4_b_rate_name_collection,

    AVG(CASE WHEN period='prior4' THEN q_rate_dob_collection END) AS p4_q_rate_dob_collection,
    AVG(CASE WHEN period='prior4' THEN b_rate_dob_collection END) AS p4_b_rate_dob_collection,

    AVG(CASE WHEN period='prior4' THEN q_rate_credit_check_no_hit END) AS p4_q_rate_credit_check_no_hit,
    AVG(CASE WHEN period='prior4' THEN b_rate_credit_check_no_hit END) AS p4_b_rate_credit_check_no_hit,

    AVG(CASE WHEN period='prior4' THEN q_rate_with_credit_hit END) AS p4_q_rate_with_credit_hit,
    AVG(CASE WHEN period='prior4' THEN b_rate_with_credit_hit END) AS p4_b_rate_with_credit_hit,

    -- step counts prior4
    AVG(CASE WHEN period='prior4' THEN CAST(reached_entered_compass_calls AS DOUBLE) END)      AS p4_reached_entered_compass_calls,
    AVG(CASE WHEN period='prior4' THEN CAST(reached_moving_switching_calls AS DOUBLE) END)     AS p4_reached_moving_switching_calls,
    AVG(CASE WHEN period='prior4' THEN CAST(reached_zip_collection_calls AS DOUBLE) END)       AS p4_reached_zip_collection_calls,
    AVG(CASE WHEN period='prior4' THEN CAST(reached_address_collection_calls AS DOUBLE) END)   AS p4_reached_address_collection_calls,
    AVG(CASE WHEN period='prior4' THEN CAST(reached_ercot_check_calls AS DOUBLE) END)          AS p4_reached_ercot_check_calls,
    AVG(CASE WHEN period='prior4' THEN CAST(reached_name_collection_calls AS DOUBLE) END)      AS p4_reached_name_collection_calls,
    AVG(CASE WHEN period='prior4' THEN CAST(reached_dob_collection_calls AS DOUBLE) END)       AS p4_reached_dob_collection_calls,
    AVG(CASE WHEN period='prior4' THEN CAST(reached_credit_check_calls AS DOUBLE) END)         AS p4_reached_credit_check_calls,

    AVG(CASE WHEN period='prior4' THEN CAST(q_at_initial_question_calls AS DOUBLE) END)        AS p4_q_at_initial_question_calls,
    AVG(CASE WHEN period='prior4' THEN CAST(b_at_initial_question_calls AS DOUBLE) END)        AS p4_b_at_initial_question_calls,

    AVG(CASE WHEN period='prior4' THEN CAST(q_at_moving_switching_calls AS DOUBLE) END)        AS p4_q_at_moving_switching_calls,
    AVG(CASE WHEN period='prior4' THEN CAST(b_at_moving_switching_calls AS DOUBLE) END)        AS p4_b_at_moving_switching_calls,

    AVG(CASE WHEN period='prior4' THEN CAST(q_at_zip_collection_calls AS DOUBLE) END)          AS p4_q_at_zip_collection_calls,
    AVG(CASE WHEN period='prior4' THEN CAST(b_at_zip_collection_calls AS DOUBLE) END)          AS p4_b_at_zip_collection_calls,

    AVG(CASE WHEN period='prior4' THEN CAST(q_at_address_collection_calls AS DOUBLE) END)      AS p4_q_at_address_collection_calls,
    AVG(CASE WHEN period='prior4' THEN CAST(b_at_address_collection_calls AS DOUBLE) END)      AS p4_b_at_address_collection_calls,

    AVG(CASE WHEN period='prior4' THEN CAST(q_at_ercot_check_calls AS DOUBLE) END)             AS p4_q_at_ercot_check_calls,
    AVG(CASE WHEN period='prior4' THEN CAST(b_at_ercot_check_calls AS DOUBLE) END)             AS p4_b_at_ercot_check_calls,

    AVG(CASE WHEN period='prior4' THEN CAST(q_at_name_collection_calls AS DOUBLE) END)         AS p4_q_at_name_collection_calls,
    AVG(CASE WHEN period='prior4' THEN CAST(b_at_name_collection_calls AS DOUBLE) END)         AS p4_b_at_name_collection_calls,

    AVG(CASE WHEN period='prior4' THEN CAST(q_at_dob_collection_calls AS DOUBLE) END)          AS p4_q_at_dob_collection_calls,
    AVG(CASE WHEN period='prior4' THEN CAST(b_at_dob_collection_calls AS DOUBLE) END)          AS p4_b_at_dob_collection_calls,

    AVG(CASE WHEN period='prior4' THEN CAST(q_at_credit_check_no_hit_calls AS DOUBLE) END)     AS p4_q_at_credit_check_no_hit_calls,
    AVG(CASE WHEN period='prior4' THEN CAST(b_at_credit_check_no_hit_calls AS DOUBLE) END)     AS p4_b_at_credit_check_no_hit_calls,

    AVG(CASE WHEN period='prior4' THEN CAST(q_with_credit_hit_calls AS DOUBLE) END)            AS p4_q_with_credit_hit_calls,
    AVG(CASE WHEN period='prior4' THEN CAST(b_with_credit_hit_calls AS DOUBLE) END)            AS p4_b_with_credit_hit_calls,

    AVG(CASE WHEN period='prior4' THEN queue_to_ivr_rate END) AS p4_queue_to_ivr_rate,
    AVG(CASE WHEN period='prior4' THEN CAST(ivr_completed_calls AS DOUBLE) END) AS p4_ivr_completed_calls,

    AVG(CASE WHEN period='prior4' THEN transactional_conversion_rate END) AS p4_transactional_conversion_rate,

    AVG(CASE WHEN period='prior4' THEN CAST(total_revenue AS DOUBLE) END) AS p4_total_revenue,
    AVG(CASE WHEN period='prior4' THEN revenue_per_order END) AS p4_revenue_per_order,
    AVG(CASE WHEN period='prior4' THEN revenue_per_net_call END) AS p4_revenue_per_net_call,
    AVG(CASE WHEN period='prior4' THEN revenue_per_gross_call END) AS p4_revenue_per_gross_call,

    -- yesterday values
    MAX(CASE WHEN period='yesterday' THEN CAST(gross_calls AS DOUBLE) END) AS y_gross_calls,
    MAX(CASE WHEN period='yesterday' THEN entered_compass_rate END) AS y_entered_compass_rate,
    MAX(CASE WHEN period='yesterday' THEN CAST(entered_compass_calls AS DOUBLE) END) AS y_entered_compass_calls,
    MAX(CASE WHEN period='yesterday' THEN compass_completion_rate END) AS y_compass_completion_rate,

    MAX(CASE WHEN period='yesterday' THEN any_bot_call_rate_compass END) AS y_any_bot_call_rate_compass,
    MAX(CASE WHEN period='yesterday' THEN CAST(non_bot_compass_calls AS DOUBLE) END) AS y_non_bot_compass_calls,

    MAX(CASE WHEN period='yesterday' THEN queue_to_gross_rate END) AS y_queue_to_gross_rate,
    MAX(CASE WHEN period='yesterday' THEN CAST(queue_calls AS DOUBLE) END) AS y_queue_calls,
    MAX(CASE WHEN period='yesterday' THEN abandonment_rate END) AS y_abandonment_rate,
    MAX(CASE WHEN period='yesterday' THEN CAST(net_calls AS DOUBLE) END) AS y_net_calls,

    MAX(CASE WHEN period='yesterday' THEN contact_rate END) AS y_contact_rate,
    MAX(CASE WHEN period='yesterday' THEN credit_rate END) AS y_credit_rate,
    MAX(CASE WHEN period='yesterday' THEN net_conversion_rate END) AS y_net_conversion_rate,
    MAX(CASE WHEN period='yesterday' THEN CAST(orders AS DOUBLE) END) AS y_orders,
    MAX(CASE WHEN period='yesterday' THEN gross_conversion_rate END) AS y_gross_conversion_rate,

    MAX(CASE WHEN period='yesterday' THEN ucc_api_call_rate END) AS y_ucc_api_call_rate,
    MAX(CASE WHEN period='yesterday' THEN ucc_api_success_given_call_rate END) AS y_ucc_api_success_given_call_rate,
    MAX(CASE WHEN period='yesterday' THEN ucc_hit_given_success_rate END) AS y_ucc_hit_given_success_rate,

    MAX(CASE WHEN period='yesterday' THEN q_rate_initial_question END) AS y_q_rate_initial_question,
    MAX(CASE WHEN period='yesterday' THEN b_rate_initial_question END) AS y_b_rate_initial_question,

    MAX(CASE WHEN period='yesterday' THEN q_rate_moving_switching END) AS y_q_rate_moving_switching,
    MAX(CASE WHEN period='yesterday' THEN b_rate_moving_switching END) AS y_b_rate_moving_switching,

    MAX(CASE WHEN period='yesterday' THEN q_rate_zip_collection END) AS y_q_rate_zip_collection,
    MAX(CASE WHEN period='yesterday' THEN b_rate_zip_collection END) AS y_b_rate_zip_collection,

    MAX(CASE WHEN period='yesterday' THEN q_rate_address_collection END) AS y_q_rate_address_collection,
    MAX(CASE WHEN period='yesterday' THEN b_rate_address_collection END) AS y_b_rate_address_collection,

    MAX(CASE WHEN period='yesterday' THEN q_rate_ercot_check END) AS y_q_rate_ercot_check,
    MAX(CASE WHEN period='yesterday' THEN b_rate_ercot_check END) AS y_b_rate_ercot_check,

    MAX(CASE WHEN period='yesterday' THEN q_rate_name_collection END) AS y_q_rate_name_collection,
    MAX(CASE WHEN period='yesterday' THEN b_rate_name_collection END) AS y_b_rate_name_collection,

    MAX(CASE WHEN period='yesterday' THEN q_rate_dob_collection END) AS y_q_rate_dob_collection,
    MAX(CASE WHEN period='yesterday' THEN b_rate_dob_collection END) AS y_b_rate_dob_collection,

    MAX(CASE WHEN period='yesterday' THEN q_rate_credit_check_no_hit END) AS y_q_rate_credit_check_no_hit,
    MAX(CASE WHEN period='yesterday' THEN b_rate_credit_check_no_hit END) AS y_b_rate_credit_check_no_hit,

    MAX(CASE WHEN period='yesterday' THEN q_rate_with_credit_hit END) AS y_q_rate_with_credit_hit,
    MAX(CASE WHEN period='yesterday' THEN b_rate_with_credit_hit END) AS y_b_rate_with_credit_hit

  FROM daily
  CROSS JOIN params p
),

final AS (

  /* -------------------------
     VOLUME + ENTRY
     ------------------------- */

  SELECT  1 AS sort_key, 'Gross Calls' AS KPI,
          p4_gross_calls AS P4WA, y_gross_calls AS y_value,
          try_divide(y_gross_calls - p4_gross_calls, NULLIF(p4_gross_calls,0)) AS Delta,
          p4_gross_calls AS P4WA_calls,
          y_gross_calls  AS Yesterday_calls,
          (y_gross_calls - p4_gross_calls) AS Call_Count_Delta
  FROM period_agg

  UNION ALL SELECT  2, 'Entered Compass Calls (Count)',
          p4_entered_compass_calls, y_entered_compass_calls,
          try_divide(y_entered_compass_calls - p4_entered_compass_calls, NULLIF(p4_entered_compass_calls,0)),
          p4_entered_compass_calls,
          y_entered_compass_calls,
          (y_entered_compass_calls - p4_entered_compass_calls)
  FROM period_agg

  UNION ALL SELECT  3, 'Entered Compass Rate',
          p4_entered_compass_rate, y_entered_compass_rate,
          try_divide(y_entered_compass_rate - p4_entered_compass_rate, NULLIF(p4_entered_compass_rate,0)),
          p4_entered_compass_calls,
          y_entered_compass_calls,
          (y_entered_compass_calls - p4_entered_compass_calls)
  FROM period_agg


  /* -------------------------
     BOT
     ------------------------- */

  UNION ALL SELECT  4, 'Any Bot Call Rate (Compass)',
          p4_any_bot_call_rate_compass, y_any_bot_call_rate_compass,
          try_divide(y_any_bot_call_rate_compass - p4_any_bot_call_rate_compass, NULLIF(p4_any_bot_call_rate_compass,0)),
          p4_entered_compass_calls,
          y_entered_compass_calls,
          (y_entered_compass_calls - p4_entered_compass_calls)
  FROM period_agg

  UNION ALL SELECT  5, 'Non-Bot Compass Calls (Count)',
          p4_non_bot_compass_calls, y_non_bot_compass_calls,
          try_divide(y_non_bot_compass_calls - p4_non_bot_compass_calls, NULLIF(p4_non_bot_compass_calls,0)),
          p4_non_bot_compass_calls,
          y_non_bot_compass_calls,
          (y_non_bot_compass_calls - p4_non_bot_compass_calls)
  FROM period_agg


    /* -------------------------
     STEP-LEVEL QUEUE RATES
     ------------------------- */

  UNION ALL SELECT  30, 'Queue Rate – Initial Question',
          p4_q_rate_initial_question, y_q_rate_initial_question,
          try_divide(y_q_rate_initial_question - p4_q_rate_initial_question, NULLIF(p4_q_rate_initial_question,0)),
          p4_reached_entered_compass_calls,
          y_entered_compass_calls,
          (y_entered_compass_calls - p4_reached_entered_compass_calls)
  FROM period_agg

  UNION ALL SELECT  31, 'Queue Rate – Moving / Switching',
          p4_q_rate_moving_switching, y_q_rate_moving_switching,
          try_divide(y_q_rate_moving_switching - p4_q_rate_moving_switching, NULLIF(p4_q_rate_moving_switching,0)),
          p4_reached_moving_switching_calls,
          y_entered_compass_calls,
          (y_entered_compass_calls - p4_reached_moving_switching_calls)
  FROM period_agg

  UNION ALL SELECT  32, 'Queue Rate – ZIP Collection',
          p4_q_rate_zip_collection, y_q_rate_zip_collection,
          try_divide(y_q_rate_zip_collection - p4_q_rate_zip_collection, NULLIF(p4_q_rate_zip_collection,0)),
          p4_reached_zip_collection_calls,
          y_entered_compass_calls,
          (y_entered_compass_calls - p4_reached_zip_collection_calls)
  FROM period_agg

  UNION ALL SELECT  33, 'Queue Rate – Address Collection',
          p4_q_rate_address_collection, y_q_rate_address_collection,
          try_divide(y_q_rate_address_collection - p4_q_rate_address_collection, NULLIF(p4_q_rate_address_collection,0)),
          p4_reached_address_collection_calls,
          y_entered_compass_calls,
          (y_entered_compass_calls - p4_reached_address_collection_calls)
  FROM period_agg

  UNION ALL SELECT  34, 'Queue Rate – ERCOT Check',
          p4_q_rate_ercot_check, y_q_rate_ercot_check,
          try_divide(y_q_rate_ercot_check - p4_q_rate_ercot_check, NULLIF(p4_q_rate_ercot_check,0)),
          p4_reached_ercot_check_calls,
          y_entered_compass_calls,
          (y_entered_compass_calls - p4_reached_ercot_check_calls)
  FROM period_agg

  UNION ALL SELECT  35, 'Queue Rate – Name Collection',
          p4_q_rate_name_collection, y_q_rate_name_collection,
          try_divide(y_q_rate_name_collection - p4_q_rate_name_collection, NULLIF(p4_q_rate_name_collection,0)),
          p4_reached_name_collection_calls,
          y_entered_compass_calls,
          (y_entered_compass_calls - p4_reached_name_collection_calls)
  FROM period_agg

  UNION ALL SELECT  36, 'Queue Rate – DOB Collection',
          p4_q_rate_dob_collection, y_q_rate_dob_collection,
          try_divide(y_q_rate_dob_collection - p4_q_rate_dob_collection, NULLIF(p4_q_rate_dob_collection,0)),
          p4_reached_dob_collection_calls,
          y_entered_compass_calls,
          (y_entered_compass_calls - p4_reached_dob_collection_calls)
  FROM period_agg

  UNION ALL SELECT  37, 'Queue Rate – Credit Check (No Hit)',
          p4_q_rate_credit_check_no_hit, y_q_rate_credit_check_no_hit,
          try_divide(y_q_rate_credit_check_no_hit - p4_q_rate_credit_check_no_hit, NULLIF(p4_q_rate_credit_check_no_hit,0)),
          p4_reached_credit_check_calls,
          y_entered_compass_calls,
          (y_entered_compass_calls - p4_reached_credit_check_calls)
  FROM period_agg

  UNION ALL SELECT  38, 'Queue Rate – Credit Check (Hit)',
          p4_q_rate_with_credit_hit, y_q_rate_with_credit_hit,
          try_divide(y_q_rate_with_credit_hit - p4_q_rate_with_credit_hit, NULLIF(p4_q_rate_with_credit_hit,0)),
          p4_reached_credit_check_calls,
          y_entered_compass_calls,
          (y_entered_compass_calls - p4_reached_credit_check_calls)
  FROM period_agg


  /* -------------------------
     STEP-LEVEL BREAKAGE RATES
     ------------------------- */

  UNION ALL SELECT  40, 'Breakage Rate – Initial Question',
          p4_b_rate_initial_question, y_b_rate_initial_question,
          try_divide(y_b_rate_initial_question - p4_b_rate_initial_question, NULLIF(p4_b_rate_initial_question,0)),
          p4_reached_entered_compass_calls,
          y_entered_compass_calls,
          (y_entered_compass_calls - p4_reached_entered_compass_calls)
  FROM period_agg

  UNION ALL SELECT  41, 'Breakage Rate – Moving / Switching',
          p4_b_rate_moving_switching, y_b_rate_moving_switching,
          try_divide(y_b_rate_moving_switching - p4_b_rate_moving_switching, NULLIF(p4_b_rate_moving_switching,0)),
          p4_reached_moving_switching_calls,
          y_entered_compass_calls,
          (y_entered_compass_calls - p4_reached_moving_switching_calls)
  FROM period_agg

  UNION ALL SELECT  42, 'Breakage Rate – ZIP Collection',
          p4_b_rate_zip_collection, y_b_rate_zip_collection,
          try_divide(y_b_rate_zip_collection - p4_b_rate_zip_collection, NULLIF(p4_b_rate_zip_collection,0)),
          p4_reached_zip_collection_calls,
          y_entered_compass_calls,
          (y_entered_compass_calls - p4_reached_zip_collection_calls)
  FROM period_agg

  UNION ALL SELECT  43, 'Breakage Rate – Address Collection',
          p4_b_rate_address_collection, y_b_rate_address_collection,
          try_divide(y_b_rate_address_collection - p4_b_rate_address_collection, NULLIF(p4_b_rate_address_collection,0)),
          p4_reached_address_collection_calls,
          y_entered_compass_calls,
          (y_entered_compass_calls - p4_reached_address_collection_calls)
  FROM period_agg

  UNION ALL SELECT  44, 'Breakage Rate – ERCOT Check',
          p4_b_rate_ercot_check, y_b_rate_ercot_check,
          try_divide(y_b_rate_ercot_check - p4_b_rate_ercot_check, NULLIF(p4_b_rate_ercot_check,0)),
          p4_reached_ercot_check_calls,
          y_entered_compass_calls,
          (y_entered_compass_calls - p4_reached_ercot_check_calls)
  FROM period_agg

  UNION ALL SELECT  45, 'Breakage Rate – Name Collection',
          p4_b_rate_name_collection, y_b_rate_name_collection,
          try_divide(y_b_rate_name_collection - p4_b_rate_name_collection, NULLIF(p4_b_rate_name_collection,0)),
          p4_reached_name_collection_calls,
          y_entered_compass_calls,
          (y_entered_compass_calls - p4_reached_name_collection_calls)
  FROM period_agg

  UNION ALL SELECT  46, 'Breakage Rate – DOB Collection',
          p4_b_rate_dob_collection, y_b_rate_dob_collection,
          try_divide(y_b_rate_dob_collection - p4_b_rate_dob_collection, NULLIF(p4_b_rate_dob_collection,0)),
          p4_reached_dob_collection_calls,
          y_entered_compass_calls,
          (y_entered_compass_calls - p4_reached_dob_collection_calls)
  FROM period_agg

  UNION ALL SELECT  47, 'Breakage Rate – Credit Check (No Hit)',
          p4_b_rate_credit_check_no_hit, y_b_rate_credit_check_no_hit,
          try_divide(y_b_rate_credit_check_no_hit - p4_b_rate_credit_check_no_hit, NULLIF(p4_b_rate_credit_check_no_hit,0)),
          p4_reached_credit_check_calls,
          y_entered_compass_calls,
          (y_entered_compass_calls - p4_reached_credit_check_calls)
  FROM period_agg

  UNION ALL SELECT  48, 'Breakage Rate – Credit Check (Hit)',
          p4_b_rate_with_credit_hit, y_b_rate_with_credit_hit,
          try_divide(y_b_rate_with_credit_hit - p4_b_rate_with_credit_hit, NULLIF(p4_b_rate_with_credit_hit,0)),
          p4_reached_credit_check_calls,
          y_entered_compass_calls,
          (y_entered_compass_calls - p4_reached_credit_check_calls)
  FROM period_agg

  /* -------------------------
     QUEUE / ABANDON
     ------------------------- */

  UNION ALL SELECT  50, 'Queue Calls',
          p4_queue_calls, y_queue_calls,
          try_divide(y_queue_calls - p4_queue_calls, NULLIF(p4_queue_calls,0)),
          p4_gross_calls,
          y_gross_calls,
          (y_queue_calls - p4_queue_calls)
  FROM period_agg

  UNION ALL SELECT  51, 'Queue to Gross Rate',
          p4_queue_to_gross_rate, y_queue_to_gross_rate,
          try_divide(y_queue_to_gross_rate - p4_queue_to_gross_rate, NULLIF(p4_queue_to_gross_rate,0)),
          p4_gross_calls,
          y_gross_calls,
          (y_gross_calls - p4_gross_calls)
  FROM period_agg

  UNION ALL SELECT  52, 'Abandonment Rate',
          p4_abandonment_rate, y_abandonment_rate,
          try_divide(y_abandonment_rate - p4_abandonment_rate, NULLIF(p4_abandonment_rate,0)),
          p4_queue_calls,
          y_queue_calls,
          (y_queue_calls - p4_queue_calls)
  FROM period_agg


  /* -------------------------
     NET / SALES
     ------------------------- */

  UNION ALL SELECT  60, 'Net Calls',
          p4_net_calls, y_net_calls,
          try_divide(y_net_calls - p4_net_calls, NULLIF(p4_net_calls,0)),
          p4_net_calls,
          y_net_calls,
          (y_net_calls - p4_net_calls)
  FROM period_agg

  UNION ALL SELECT  61, 'Contact Rate',
          p4_contact_rate, y_contact_rate,
          try_divide(y_contact_rate - p4_contact_rate, NULLIF(p4_contact_rate,0)),
          p4_net_calls,
          y_net_calls,
          (y_net_calls - p4_net_calls)
  FROM period_agg

  UNION ALL SELECT  62, 'Credit Rate',
          p4_credit_rate, y_credit_rate,
          try_divide(y_credit_rate - p4_credit_rate, NULLIF(p4_credit_rate,0)),
          p4_net_calls,
          y_net_calls,
          (y_net_calls - p4_net_calls)
  FROM period_agg

  UNION ALL SELECT  63, 'Net Conversion Rate',
          p4_net_conversion_rate, y_net_conversion_rate,
          try_divide(y_net_conversion_rate - p4_net_conversion_rate, NULLIF(p4_net_conversion_rate,0)),
          p4_net_calls,
          y_net_calls,
          (y_net_calls - p4_net_calls)
  FROM period_agg

  UNION ALL SELECT  64, 'Orders',
          p4_orders, y_orders,
          try_divide(y_orders - p4_orders, NULLIF(p4_orders,0)),
          p4_net_calls,
          y_net_calls,
          (y_orders - p4_orders)
  FROM period_agg

  UNION ALL SELECT  65, 'Gross Conversion Rate',
          p4_gross_conversion_rate, y_gross_conversion_rate,
          try_divide(y_gross_conversion_rate - p4_gross_conversion_rate, NULLIF(p4_gross_conversion_rate,0)),
          p4_gross_calls,
          y_gross_calls,
          (y_gross_calls - p4_gross_calls)
  FROM period_agg

)

SELECT
  sort_key,
  KPI,
  P4WA,
  y_value AS `Yesterday`,
  Delta,
  P4WA_calls,
  Yesterday_calls,
  Call_Count_Delta
FROM final
ORDER BY sort_key
;


"""

    # Optionally preview the SQL (first N chars)
    if debug_print_sql:
        print(f"--- compute_kpis() will create/replace temp view: {out_view} ---")
        # print a short preview so notebook output remains readable
        print(full_funnel_kpis_sql[:3000])
        print("--- end SQL preview ---\n")

    try:
        # Execute the embedded SQL which must create the {out_view} temp view
        spark.sql(full_funnel_kpis_sql)

        # Return the results
        return spark.sql(f"SELECT * FROM {out_view}")

    except Exception as e:
        print("ERROR running compute_kpis():", type(e).__name__, str(e))
        try:
            print("\n--- SQL (first 8000 chars) ---")
            print(full_funnel_kpis_sql[:8000])
            print("\n--- SQL (last 8000 chars) ---")
            print(full_funnel_kpis_sql[-8000:])
        except Exception:
            pass
        raise


### Converting SQL Output to Structured JSON

In [0]:
import re
from typing import Any, Dict, List, Optional, Union

# -----------------------------
# helpers (unchanged)
# -----------------------------
def _to_snake(s: str) -> str:
    s = s.strip().lower()
    s = s.replace("–", "-").replace("—", "-")
    s = re.sub(r"[/%()]", " ", s)
    s = re.sub(r"[^a-z0-9]+", "_", s)
    s = re.sub(r"_+", "_", s).strip("_")
    return s

def _safe_float(x: Any) -> Optional[float]:
    if x is None:
        return None
    try:
        return float(x)
    except Exception:
        return None

def _is_breakdown_kpi(kpi_name: str, delim: str = " - ") -> bool:
    norm = kpi_name.replace("–", "-").replace("—", "-")
    return " - " in norm

def _split_breakdown(kpi_name: str):
    norm = kpi_name.replace("–", "-").replace("—", "-")
    parts = [p.strip() for p in norm.split("-", 1)]
    if len(parts) != 2:
        return (kpi_name.strip(), None)
    return (parts[0].strip(), parts[1].strip())


# -----------------------------
# SPEED IMPROVED VERSION
# -----------------------------
def kpi_tables_to_json(
    names: List[str],
    tables: List[Union[str, Any]],  # str table/view name OR Spark DataFrame
    *,
    kpi_col: str = "KPI",
    current_col: str = "Yesterday",
    baseline_col: str = "P4WA",
    delta_col: str = "Delta",
    current_calls_col: str = "Yesterday_calls",
    baseline_calls_col: str = "P4WA_calls",
    call_delta_col: str = "Call_Count_Delta",
    breakdown_reason_key: str = "reason",
    prefer_precomputed_delta: bool = True,
    # Speed / safety knobs
    max_collect_rows: int = 2000,          # fail fast if a KPI DF is accidentally huge
    use_collect_instead_of_pandas: bool = True,  # avoid toPandas overhead
) -> Dict[str, Any]:
    """
    Key speed improvements:
      1) Uses Spark .collect() instead of .toPandas() by default (less overhead).
      2) Optional guardrail (max_collect_rows) prevents accidental huge driver collects.
      3) Selects only needed columns (avoids wide-row collection).
    """
    if len(names) != len(tables):
        raise ValueError(f"names and tables must be same length. Got {len(names)} vs {len(tables)}")

    out: Dict[str, Any] = {}

    for section_name, tbl in zip(names, tables):
        sdf = spark.table(tbl) if isinstance(tbl, str) else tbl

        cols = set(sdf.columns)
        required = {kpi_col, current_col, baseline_col}
        missing = required - cols
        if missing:
            raise ValueError(f"Section '{section_name}' missing required columns: {sorted(missing)}")

        # Only pull what we actually need
        wanted = [kpi_col, current_col, baseline_col]
        if delta_col in cols:
            wanted.append(delta_col)
        if current_calls_col in cols:
            wanted.append(current_calls_col)
        if baseline_calls_col in cols:
            wanted.append(baseline_calls_col)
        if call_delta_col in cols:
            wanted.append(call_delta_col)

        sdf_small = sdf.select(*wanted)

        # Guardrail: KPI outputs should be small; avoid accidental "collect millions"
        n = sdf_small.count()
        if n > max_collect_rows:
            raise ValueError(
                f"Section '{section_name}' has {n:,} rows; refusing to collect to driver. "
                f"Check compute_kpis output (it should be a small KPI table)."
            )

        section_payload = {"metrics": {}, "breakdowns": {}}

        if use_collect_instead_of_pandas:
            rows = sdf_small.collect()
            for r in rows:
                kpi_name = str(r[kpi_col]).strip() if r[kpi_col] is not None else ""
                if not kpi_name:
                    continue

                current_val = _safe_float(r[current_col])
                baseline_val = _safe_float(r[baseline_col])

                pre_delta = _safe_float(r[delta_col]) if (delta_col in cols) else None

                delta_abs = None
                delta_pct = None
                if current_val is not None and baseline_val is not None:
                    delta_abs = current_val - baseline_val
                    if baseline_val != 0:
                        delta_pct = (current_val - baseline_val) / baseline_val

                delta_pct_precomputed = pre_delta if (prefer_precomputed_delta and pre_delta is not None) else None

                current_calls = _safe_float(r[current_calls_col]) if (current_calls_col in cols) else None
                baseline_calls = _safe_float(r[baseline_calls_col]) if (baseline_calls_col in cols) else None
                calls_delta = _safe_float(r[call_delta_col]) if (call_delta_col in cols) else None

                if _is_breakdown_kpi(kpi_name):
                    base, reason = _split_breakdown(kpi_name)
                    base_key = _to_snake(base)

                    section_payload["breakdowns"].setdefault(base_key, [])

                    delta_pct_points = None
                    if current_val is not None and baseline_val is not None:
                        delta_pct_points = current_val - baseline_val

                    item = {
                        breakdown_reason_key: reason,
                        "current": current_val,
                        "weekday_baseline_mean": baseline_val,
                        "delta_pct_points": delta_pct_points,
                        "kpi_label": kpi_name
                    }

                    if current_calls is not None:
                        item["current_calls"] = current_calls
                    if baseline_calls is not None:
                        item["weekday_baseline_calls"] = baseline_calls
                    if calls_delta is not None:
                        item["calls_delta"] = calls_delta
                    if delta_pct_precomputed is not None:
                        item["delta_pct_precomputed"] = delta_pct_precomputed

                    section_payload["breakdowns"][base_key].append(item)

                else:
                    metric_key = _to_snake(kpi_name)
                    metric_obj = {
                        "current": current_val,
                        "weekday_baseline_mean": baseline_val,
                        "delta_abs": delta_abs,
                        "delta_pct": delta_pct,
                        "kpi_label": kpi_name
                    }

                    if current_calls is not None:
                        metric_obj["current_calls"] = current_calls
                    if baseline_calls is not None:
                        metric_obj["weekday_baseline_calls"] = baseline_calls
                    if calls_delta is not None:
                        metric_obj["calls_delta"] = calls_delta
                    if delta_pct_precomputed is not None:
                        metric_obj["delta_pct_precomputed"] = delta_pct_precomputed

                    section_payload["metrics"][metric_key] = metric_obj

        else:
            # fallback to pandas (kept for completeness)
            pdf = sdf_small.toPandas()
            for _, row in pdf.iterrows():
                kpi_name = str(row.get(kpi_col, "")).strip()
                if not kpi_name:
                    continue

                current_val = _safe_float(row.get(current_col))
                baseline_val = _safe_float(row.get(baseline_col))
                pre_delta = _safe_float(row.get(delta_col)) if (delta_col in cols) else None

                delta_abs = None
                delta_pct = None
                if current_val is not None and baseline_val is not None:
                    delta_abs = current_val - baseline_val
                    if baseline_val != 0:
                        delta_pct = (current_val - baseline_val) / baseline_val

                delta_pct_precomputed = pre_delta if (prefer_precomputed_delta and pre_delta is not None) else None

                current_calls = _safe_float(row.get(current_calls_col)) if (current_calls_col in cols) else None
                baseline_calls = _safe_float(row.get(baseline_calls_col)) if (baseline_calls_col in cols) else None
                calls_delta = _safe_float(row.get(call_delta_col)) if (call_delta_col in cols) else None

                if _is_breakdown_kpi(kpi_name):
                    base, reason = _split_breakdown(kpi_name)
                    base_key = _to_snake(base)
                    section_payload["breakdowns"].setdefault(base_key, [])

                    delta_pct_points = None
                    if current_val is not None and baseline_val is not None:
                        delta_pct_points = current_val - baseline_val

                    item = {
                        breakdown_reason_key: reason,
                        "current": current_val,
                        "weekday_baseline_mean": baseline_val,
                        "delta_pct_points": delta_pct_points,
                        "kpi_label": kpi_name
                    }

                    if current_calls is not None:
                        item["current_calls"] = current_calls
                    if baseline_calls is not None:
                        item["weekday_baseline_calls"] = baseline_calls
                    if calls_delta is not None:
                        item["calls_delta"] = calls_delta
                    if delta_pct_precomputed is not None:
                        item["delta_pct_precomputed"] = delta_pct_precomputed

                    section_payload["breakdowns"][base_key].append(item)

                else:
                    metric_key = _to_snake(kpi_name)
                    metric_obj = {
                        "current": current_val,
                        "weekday_baseline_mean": baseline_val,
                        "delta_abs": delta_abs,
                        "delta_pct": delta_pct,
                        "kpi_label": kpi_name
                    }

                    if current_calls is not None:
                        metric_obj["current_calls"] = current_calls
                    if baseline_calls is not None:
                        metric_obj["weekday_baseline_calls"] = baseline_calls
                    if calls_delta is not None:
                        metric_obj["calls_delta"] = calls_delta
                    if delta_pct_precomputed is not None:
                        metric_obj["delta_pct_precomputed"] = delta_pct_precomputed

                    section_payload["metrics"][metric_key] = metric_obj

        out[section_name] = section_payload

    return out


### JSON Filtering - Removing Unimportant Metrics Before API Call

In [0]:
import math
from typing import Any, Dict, List, Optional, Tuple

def filter_and_compact_kpi_json(
    kpi_json: Dict[str, Any],
    *,
    # -----------------------
    # Filtering thresholds
    # -----------------------
    # Drop whole sections with tiny volume (based on section gross calls)
    min_section_gross_calls: float = 0.0,

    # Metric-level volume gates (uses metric["current_calls"] when present, else section gross)
    min_metric_current_calls: float = 25.0,

    # Metric impact gates (keeps metric if ANY of these pass)
    min_abs_delta_pct: float = 0.03,          # e.g. 0.03 == 3% relative change
    min_abs_delta_abs: float = 0.0,           # optional for count KPIs; 0 disables if left at 0
    min_abs_delta_pp: float = 0.01,           # for rate-like metrics when delta_abs is “pp” in raw units (0.01==1pp)

    # Always keep these metric keys (snake keys in your JSON), regardless of thresholds
    always_keep_metrics: Tuple[str, ...] = (
        "gross_calls",
        "entered_compass_calls_count",
        "entered_compass_rate",
        "queue_calls",
        "queue_to_gross_rate",
        "abandonment_rate",
        "net_calls",
        "orders",
        "gross_conversion_rate",
    ),

    # Breakdowns: keep top-N by absolute delta pct points (pp), after filtering
    breakdown_top_n: int = 8,
    min_breakdown_abs_delta_pp: float = 0.01,  # 1pp
    min_breakdown_current_calls: float = 25.0,

    # -----------------------
    # Compaction controls
    # -----------------------
    keep_metric_fields: Tuple[str, ...] = ("current", "weekday_baseline_mean", "delta_abs", "delta_pct", "current_calls"),
    keep_breakdown_fields: Tuple[str, ...] = ("reason", "current", "weekday_baseline_mean", "delta_pct_points", "current_calls"),
    drop_filter_summary: bool = True,
    drop_empty_breakdowns: bool = True,
    drop_zero_volume_metrics: bool = True,
    round_digits: Optional[int] = 6,
) -> Dict[str, Any]:
    """
    One-stop function:
      1) Filters out low-impact metrics + low-volume metrics + low-value breakdown items
      2) Compacts to a minimal JSON payload for LLM prompting
      3) Cleans NaN/Inf so output is valid JSON

    Assumes input structure like:
      {
        "<section>": {
          "metrics": { "<metric_key>": { ... } },
          "breakdowns": { "<base_key>": [ { ... }, ... ] },
          "filter_summary": { ... }   # optional
        },
        ...
      }
    """

    def _is_nan(x: Any) -> bool:
        return isinstance(x, float) and math.isnan(x)

    def _clean(x: Any) -> Any:
        if x is None:
            return None
        if _is_nan(x):
            return None
        if isinstance(x, float) and math.isinf(x):
            return None
        return x

    def _round(v: Any) -> Any:
        v = _clean(v)
        if v is None:
            return None
        if round_digits is None:
            return v
        if isinstance(v, float):
            return round(v, round_digits)
        return v

    def _get_float(o: Dict[str, Any], key: str) -> Optional[float]:
        v = _clean(o.get(key))
        try:
            return float(v) if v is not None else None
        except Exception:
            return None

    def _metric_volume(metric_obj: Dict[str, Any], section_gross: Optional[float]) -> Optional[float]:
        # Prefer metric current_calls, else section gross
        mc = _get_float(metric_obj, "current_calls")
        if mc is not None:
            return mc
        return section_gross

    def _metric_keep(metric_key: str, metric_obj: Dict[str, Any], section_gross: Optional[float]) -> bool:
        if metric_key in always_keep_metrics:
            return True

        cur = _get_float(metric_obj, "current")
        if cur is None:
            return False

        if drop_zero_volume_metrics:
            vol = _metric_volume(metric_obj, section_gross)
            if vol is not None and vol == 0:
                return False

        vol = _metric_volume(metric_obj, section_gross)
        if vol is not None and vol < min_metric_current_calls:
            return False

        # Impacts
        d_pct = _get_float(metric_obj, "delta_pct")
        d_abs = _get_float(metric_obj, "delta_abs")  # could be pp for rates or abs change for counts

        passes = False
        if d_pct is not None and abs(d_pct) >= min_abs_delta_pct:
            passes = True
        if (not passes) and (min_abs_delta_abs > 0) and (d_abs is not None) and abs(d_abs) >= min_abs_delta_abs:
            passes = True
        if (not passes) and (d_abs is not None) and abs(d_abs) >= min_abs_delta_pp:
            # For rates, delta_abs is typically pp in raw units (0.01 == 1pp)
            passes = True

        return passes

    def _compact_metric(metric_obj: Dict[str, Any]) -> Dict[str, Any]:
        o = metric_obj or {}
        outm = {}
        for f in keep_metric_fields:
            if f in o:
                outm[f] = _round(o.get(f))
        # drop nulls
        return {k: v for k, v in outm.items() if v is not None}

    def _compact_breakdown_item(item: Dict[str, Any]) -> Dict[str, Any]:
        it = item or {}
        outb = {}
        for f in keep_breakdown_fields:
            if f in it:
                outb[f] = _round(it.get(f))
        return {k: v for k, v in outb.items() if v is not None}

    filtered_compacted: Dict[str, Any] = {}

    for section_name, section in (kpi_json or {}).items():
        section = section or {}
        metrics_in = section.get("metrics", {}) or {}
        breakdowns_in = section.get("breakdowns", {}) or {}

        # Section gross calls (best-effort)
        section_gross = None
        if "gross_calls" in metrics_in:
            section_gross = _get_float(metrics_in["gross_calls"], "current_calls")
            if section_gross is None:
                section_gross = _get_float(metrics_in["gross_calls"], "current")

        if section_gross is not None and section_gross < min_section_gross_calls:
            continue

        # ---- Metrics: filter + compact ----
        metrics_out: Dict[str, Any] = {}
        for metric_key, metric_obj in metrics_in.items():
            metric_obj = metric_obj or {}

            if not _metric_keep(metric_key, metric_obj, section_gross):
                continue

            cm = _compact_metric(metric_obj)
            if not cm:
                continue

            # Optional final drop (if current_calls exists and is 0)
            if drop_zero_volume_metrics and ("current_calls" in cm) and (cm["current_calls"] == 0):
                continue

            metrics_out[metric_key] = cm

        # If a section ends up empty, skip it
        if not metrics_out:
            continue

        # ---- Breakdowns: filter + rank + compact ----
        breakdowns_out: Dict[str, List[Dict[str, Any]]] = {}
        for base_key, items in breakdowns_in.items():
            kept: List[Tuple[float, Dict[str, Any]]] = []

            for item in (items or []):
                item = item or {}
                cur = _get_float(item, "current")
                if cur is None:
                    continue

                # volume gate
                bvol = _get_float(item, "current_calls")
                if bvol is not None and bvol < min_breakdown_current_calls:
                    continue
                if drop_zero_volume_metrics and bvol is not None and bvol == 0:
                    continue

                dpp = _get_float(item, "delta_pct_points")
                if dpp is None or abs(dpp) < min_breakdown_abs_delta_pp:
                    continue

                kept.append((abs(dpp), item))

            if not kept:
                continue

            kept.sort(key=lambda x: x[0], reverse=True)
            top_items = [it for _, it in kept[:breakdown_top_n]]

            compacted_items = []
            for it in top_items:
                cb = _compact_breakdown_item(it)
                if cb:
                    compacted_items.append(cb)

            if compacted_items:
                breakdowns_out[base_key] = compacted_items

        section_out: Dict[str, Any] = {"metrics": metrics_out}
        if not (drop_empty_breakdowns and not breakdowns_out):
            section_out["breakdowns"] = breakdowns_out

        if not drop_filter_summary and "filter_summary" in section:
            fs = section.get("filter_summary") or {}
            section_out["filter_summary"] = {k: _round(v) for k, v in fs.items() if _round(v) is not None}

        filtered_compacted[section_name] = section_out

    return filtered_compacted


### OpenAI API Calls

### Main Execution

In [0]:
from tqdm import tqdm
import time
import json

allowed_buckets = {
    "Natural","Brand-Partner","Generic","Aggregator","Competitor",
    "Utility","PMax","NRG","Other Bucket"
}

t0_all = time.time()
print(f"[START] Pipeline started at {time.strftime('%Y-%m-%d %H:%M:%S')}")

section_names = []
section_tables = []

def _stamp(msg: str, t0: float):
    elapsed = time.time() - t0
    print(f"[{time.strftime('%Y-%m-%d %H:%M:%S')}] {msg} | elapsed={elapsed:,.1f}s")


# -------------------
# 1) OVERALL
# -------------------
t0 = time.time()
_stamp("START overall: get_data()", t0)

calls_df_overall = get_data(
    company_id=25,
    call_direction="INBOUND",
    restrict_to_yday_and_prior4=True,
    create_temp_view=False,
    marketing_buckets=None,
    site_serp=None
)

_stamp(f"DONE overall: get_data() | rows={calls_df_overall.count():,}", t0)

t1 = time.time()
_stamp("START overall: compute_kpis()", t1)

kpi_df_overall = compute_kpis(
    calls_df=calls_df_overall,
    source_view="calls_full_debug_overall",
    create_temp_view=True,
    debug_print_sql=False
)

_stamp(f"DONE overall: compute_kpis() | rows={kpi_df_overall.count():,}", t1)

section_names.append("overall")
section_tables.append(kpi_df_overall)


# -------------------
# 2) MARKETING BUCKETS
# -------------------
buckets_sorted = sorted(list(allowed_buckets))

t_buckets = time.time()
_stamp(f"START buckets loop | n={len(buckets_sorted)}", t_buckets)

for bucket in tqdm(buckets_sorted, desc="Marketing buckets"):
    t_bucket = time.time()
    _stamp(f"START bucket='{bucket}': get_data()", t_bucket)

    calls_df_bucket = get_data(
        company_id=25,
        call_direction="INBOUND",
        restrict_to_yday_and_prior4=True,
        create_temp_view=False,
        marketing_buckets=bucket,
        site_serp=None
    )

    _stamp(f"DONE bucket='{bucket}': get_data() | rows={calls_df_bucket.count():,}", t_bucket)

    t_bucket_kpi = time.time()
    _stamp(f"START bucket='{bucket}': compute_kpis()", t_bucket_kpi)

    kpi_df_bucket = compute_kpis(
        calls_df=calls_df_bucket,
        source_view=f"calls_full_debug_bucket_{_to_snake(bucket)}",
        create_temp_view=True,
        debug_print_sql=False
    )

    _stamp(f"DONE bucket='{bucket}': compute_kpis() | rows={kpi_df_bucket.count():,}", t_bucket_kpi)

    section_names.append(f"bucket_{_to_snake(bucket)}")
    section_tables.append(kpi_df_bucket)

_stamp("DONE buckets loop", t_buckets)


# -------------------
# 3) SITE / SERP
# -------------------
t_ss = time.time()
_stamp("START site/serp loop", t_ss)

for ss in tqdm(["Site", "SERP"], desc="Site vs SERP"):
    t_ss_item = time.time()
    _stamp(f"START {ss}: get_data()", t_ss_item)

    calls_df_ss = get_data(
        company_id=25,
        call_direction="INBOUND",
        restrict_to_yday_and_prior4=True,
        create_temp_view=False,
        marketing_buckets=None,
        site_serp=ss
    )

    _stamp(f"DONE {ss}: get_data() | rows={calls_df_ss.count():,}", t_ss_item)

    t_ss_kpi = time.time()
    _stamp(f"START {ss}: compute_kpis()", t_ss_kpi)

    kpi_df_ss = compute_kpis(
        calls_df=calls_df_ss,
        source_view=f"calls_full_debug_{_to_snake(ss)}",
        create_temp_view=True,
        debug_print_sql=False
    )

    _stamp(f"DONE {ss}: compute_kpis() | rows={kpi_df_ss.count():,}", t_ss_kpi)

    section_names.append(_to_snake(ss))  # "site" / "serp"
    section_tables.append(kpi_df_ss)

_stamp("DONE site/serp loop", t_ss)


# -------------------
# 4) JSON COMBINE
# -------------------
t_json = time.time()
_stamp(f"START JSON build | sections={len(section_names)}", t_json)

kpi_json = kpi_tables_to_json(
    names=section_names,
    tables=section_tables
)

_stamp("DONE JSON build", t_json)

print(json.dumps(kpi_json, indent=2))

_stamp("PIPELINE COMPLETE", t0_all)


21 Minutes

In [0]:
out_loosen = filter_and_compact_kpi_json(
    kpi_json,
    min_metric_current_calls=5,          # allow lower-volume metrics through
    min_abs_delta_pct=0.01,              # require only 1% change to flag
    min_breakdown_current_calls=5,       # allow smaller breakdown entries through
    breakdown_top_n=5
)
print(json.dumps(out_loosen, indent=2))